In [ ]:
import sys
import os
print(sys.path)
current_dir = os.getcwd()
TORCHLIMIX = os.path.abspath(os.path.join(current_dir, '../'))
sys.path.insert(0, TORCHLIMIX)

from statsmodels.stats.multitest import multipletests as mt
import os
import pandas as pd
import numpy as np
from numpy import log10

import matplotlib.pyplot as plt
import scipy.stats as stats

from torchlimix.plot._manhattan import manhattan
from torchlimix.plot._manhattan import get_abs_positions
from torchlimix.plot._qqplot import qqplot

In [ ]:
# Adjust the arguments dset and output_dir
dset = 'aINV'
output_dir = '...'
file_path = os.path.join(f"{output_dir}/{dset}/gwas", "log_likelihoods.csv")                             
# Load the data
if os.path.isfile(file_path):
    print(f"Loading log-likelihoods from {file_path}.")
    log_likelihoods = pd.read_csv(file_path)
else:
    raise FileNotFoundError(f"Likelihood file does not exist: {file_path}.")

print(log_likelihoods)
# Extract relevant columns
lml0 = log_likelihoods['lml0'].values
lml1 = log_likelihoods['lml1'].values
lml2 = log_likelihoods['lml2'].values
llr10 = log_likelihoods['lrt10'].values
llr20 = log_likelihoods['lrt20'].values
llr21 = log_likelihoods['lrt21'].values
df10 = log_likelihoods['df10'].values
df20 = log_likelihoods['df20'].values
df21 = log_likelihoods['df21'].values

# Check distribution of LRT values
plt.hist(llr10, bins=50, density=True, alpha=0.7, color='blue', label='LRT 10')
plt.hist(llr20, bins=50, density=True, alpha=0.7, color='green', label='LRT 20')
plt.hist(llr21, bins=50, density=True, alpha=0.7, color='yellow', label='LRT 21')
plt.xlabel('Likelihood Ratio Test Statistics', fontsize=15)
plt.ylabel('Density', fontsize=15)

plt.tick_params(axis='both', which='major', labelsize=13)  # Adjust 'labelsize' to your preferred size
plt.legend(fontsize=12)  
plt.show()
#1225.284425 

In [ ]:
# Load SNP annotation file
annot= pd.read_csv(os.path.join(f"{output_dir}/{dset}/gwas", "snp_annotation.csv"))
annot.columns = ['chrom', 'pos']
print(annot)

In [ ]:
# Correct the alpha level for multiple tests
chromosomes = annot['chrom']  # Adjust this to the actual key in your HDF5 file
positions = annot['pos'] 

result_st = pd.DataFrame({
    'chrom': chromosomes,
    'pos': positions,
    'pv': log_likelihoods['pv10'].values
})

reject_, pvals_corrected_, alphacSidak, alphacBonf = mt(log_likelihoods['pv10'].values, alpha=0.05, method='bonferroni')


manhattan(result_st, colora="#5689AC", colorb="#21334F", highlight_color="#FFA500", threshold=alphacBonf, pts_kws=None, ax=None)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
reject_, pvals_corrected_, alphacSidak, alphacBonf = mt(log_likelihoods['pv10'].values, alpha=0.05, method='bonferroni')
qqplot(log_likelihoods['pv10'].values, line='s', ax=ax, label='P-values', alpha=alphacBonf, pts_kws=dict(marker='o', color='blue'))
plt.legend(
    fontsize=12, loc='upper center', bbox_to_anchor=(0.5, 0.97)  # Adjust the y-coordinate
)
plt.show()

## Hypothesis Tests (`--test_type`)

Multi-trait LMM tests for genetic effects across P environments.

| Test Type | Null Hypothesis (H₀) | Alternative (H₁) | `--pheno_idx` |
|-----------|---------------------|------------------|-------------|
| `common` | No shared genetic effect | β_common ≠ 0 | Not needed |
| `any` | No genetic effect in any environment | At least one β_j ≠ 0 | Not needed |
| `specific` | No effect specific to environment j | β_specific[j] ≠ 0 | **Required** |
| `any_vs_common` | All effects are common (no GxE) | At least one specific effect exists | Not needed |
| `specific_vs_common` | Effect in environment j equals common | β[j] ≠ β_common | **Required** |

### Detailed Descriptions

**`common`**: Tests for persistent/shared genetic effects across all environments

**`any`**: Omnibus test for any genetic association.
- You do not have prior knowledge about the effect and want to test all environments at once

**`specific`**: Tests for environment-specific effect in a single environment.
- Requires `--pheno_idx` (0-indexed): which environment to test.
- Example: `--test_type specific --pheno_idx 2` tests environment 3.

**`any_vs_common`**: Tests for GxE interaction (any environment).
- 3 Hypothesis tests in total: $H_{10}$ tests common effect vs. null (no effect), $H_{20}$ tests any vs. null and $H_{21}$ tests the interction effect (any vs. common)
- Detects if effects differ across environments beyond a shared component.

**`specific_vs_common`**: Tests if one environment deviates from the common effect.
- Requires `--pheno_idx`: which environment to compare against common
- 3 Hypothesis tests in total: $H_{10}$ tests common effect vs. null (no effect), $H_{20}$ tests specific vs. null and $H_{21}$ tests the interction effect (specific vs. common)
- Useful for identifying environment-specific GxE for a particular condition

In [ ]:
# Any specific effect test

chromosomes = annot['chrom']  # Adjust this to the actual key in your HDF5 file
positions = annot['pos'] 

result_mt_10 = pd.DataFrame({
    'chrom': chromosomes,
    'pos': positions,
    'pv': log_likelihoods['pv10'].values
})
result_mt_20 = pd.DataFrame({
    'chrom': chromosomes,
    'pos': positions,
    'pv': log_likelihoods['pv20'].values,
})
result_mt_21 = pd.DataFrame({
    'chrom': chromosomes,
    'pos': positions,
    'pv': log_likelihoods['pv21'].values
})
reject_, pvals_corrected_, alphacSidak, alphacBonf = mt(log_likelihoods['pv21'].values, alpha=0.05, method='bonferroni')
print(result_mt_21)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the first Manhattan plot
manhattan(result_mt_10, 
          colora="#d5d5d4", 
          colorb="#d5d5d4", 
          highlight_color="#d5d5d4", 
          threshold=alphacBonf, 
          pts_kws={'marker': 'o', 'markersize': 5},  # Circle markers, 
          ax=ax)

manhattan(result_mt_20, 
          colora="#848081",  # Change to a different color for the second plot
          colorb="#848081", 
          highlight_color="#848081", 
          threshold=alphacBonf, 
          pts_kws={'marker': '*', 'markersize': 5}, 
          ax=ax)

manhattan(result_mt_21, 
          colora="#5c5859",  # Change to another different color for the third plot
          colorb="#5c5859", 
          highlight_color="#5c5859", 
          threshold=alphacBonf, 
          pts_kws={'marker': 's', 'markersize': 5}, 
          ax=ax)


# Set larger x and y labels
ax.set_xlabel("Chromosome", fontsize=14)  # Increase font size for x-axis label
ax.set_ylabel("-log10(p-value)", fontsize=14)  # Increase font size for y-axis label

ax.tick_params(axis='both', which='major', labelsize=14)  # Adjust tick label font size
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

handles = [
    plt.Line2D([0], [0], marker='o', color='w', label='Common vs. No', markerfacecolor='#5689AC', markersize=10),
    plt.Line2D([0], [0], marker='*', color='w', label='Any/Specific vs. No', markerfacecolor='#FF5733', markersize=10),
    plt.Line2D([0], [0], marker='s', color='w', label='Any/Specific vs. Common', markerfacecolor='#28A745', markersize=10)
]
#ax.legend(handles=handles, loc='upper center', ncol=3, fontsize=15) #, bbox_to_anchor=(0.5, 1.12))

# Combine all p-values to find SNPs significant in any test
sig_mask = (
    (result_mt_10['pv'] < alphacBonf) |
    (result_mt_20['pv'] < alphacBonf) |
    (result_mt_21['pv'] < alphacBonf)
)

sig_indices = np.where(sig_mask)[0]

# Compute abs_pos on the FULL data (matching what manhattan() sees)
full_df = pd.DataFrame({
    'chrom': chromosomes,
    'pos': positions,
    'pv': result_mt_10['pv'].values,  
})

transformed_full = get_abs_positions(full_df)

# Now extract positions only for significant SNPs (optional)
if len(sig_indices) > 0:
    for i, idx in enumerate(sig_indices):
        # Find this SNP in the sorted transformed data by matching chrom + pos
        mask = (
            (transformed_full["chrom"].values == str(chromosomes[idx])) &
            (transformed_full["pos"].values == int(positions[idx]))
        )
        x = float(transformed_full["abs_pos"].values[mask][0])

        min_pv = min(
            result_mt_10['pv'].values[idx],
            result_mt_20['pv'].values[idx],
            result_mt_21['pv'].values[idx],
        )
        y = -log10(min_pv)

        label = f"{chromosomes[idx]}:{positions[idx]}"

        ax.annotate(
            label,
            xy=(x, y),
            xytext=(0, 15),
            textcoords="offset points",
            fontsize=14,
            ha='center',
            rotation=45,
            arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
        )

all_pv = np.concatenate([
    result_mt_10['pv'].values,
    result_mt_20['pv'].values,
    result_mt_21['pv'].values,
])

# Keep only valid, positive p-values
valid_pv = all_pv[(all_pv > 0) & np.isfinite(all_pv)]
max_logp = -np.log10(valid_pv.min())

ax.set_ylim(0, max_logp * 1.15)
plt.show()



In [ ]:
# QQ plot for Any/Specific vs Common effect test
fig, ax = plt.subplots(figsize=(8, 8))
reject_, pvals_corrected_, alphacSidak, alphacBonf = mt(log_likelihoods['pv21'].values, alpha=0.05, method='bonferroni')
qqplot(log_likelihoods['pv21'].values, line='s', ax=ax, label='P-values', alpha=alphacBonf, pts_kws=dict(marker='o', color='blue'))
plt.legend(
    fontsize=12, loc='upper center', bbox_to_anchor=(0.5, 0.97)  # Adjust the y-coordinate
)
plt.show()

In [ ]:
# QQ plot for Any/Specific vs No effect test
fig, ax = plt.subplots(figsize=(8, 8))
reject_, pvals_corrected_, alphacSidak, alphacBonf = mt(log_likelihoods['pv20'].values, alpha=0.05, method='bonferroni')
qqplot(log_likelihoods['pv20'].values, line='s', ax=ax, label='P-values', alpha=alphacBonf, pts_kws=dict(marker='o', color='blue'))
plt.legend(
    fontsize=12, loc='upper center', bbox_to_anchor=(0.5, 0.97)  # Adjust the y-coordinate
)
plt.show()